In [1]:
import json
import numpy as np
import pandas as pd
import chromadb
from pathlib import Path
from chromadb.utils import embedding_functions
from llama_cpp import Llama, LlamaGrammar

RESULTS_DIR  = Path("../data/results")
ATTCK_DIR    = Path("../data/attck")
CHROMA_DIR   = Path("../data/chroma")
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH     = "../models/qwen2.5-3b-instruct-q4_k_m.gguf"
COMMUNITY_FILE = RESULTS_DIR / "community_assignments.csv"
TRIPLES_FILE   = RESULTS_DIR / "community_triples.json"
STIX_FILE      = ATTCK_DIR   / "enterprise-attack.json"
GRAPH_NODES    = RESULTS_DIR / "knowledge_graph_nodes.csv"
GRAPH_EDGES    = RESULTS_DIR / "knowledge_graph_edges.csv"
REPORTS_FILE   = RESULTS_DIR / "stage5_rag_reports.csv"
METRICS_FILE   = RESULTS_DIR / "stage5_rag_metrics.json"

EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K       = 5
RANDOM_SEED = 42
MAX_CHARS_PER_DOC = 400

print("Environment ready.")

Environment ready.


In [2]:
# Load data
community_df = pd.read_csv(COMMUNITY_FILE, low_memory=False)

with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

kg_edges = pd.read_csv(GRAPH_EDGES)
kg_nodes = pd.read_csv(GRAPH_NODES)

eval_df   = community_df[community_df["attck_technique_id"] != "BENIGN"].copy()
eval_cids = sorted(eval_df["community_id"].unique())

print(f"Loaded {len(community_df)} alerts across {community_df['community_id'].nunique()} communities")
print(f"Attack communities to evaluate: {len(eval_cids)}")
print(f"Knowledge graph: {len(kg_nodes)} nodes, {len(kg_edges)} edges")

Loaded 10684 alerts across 19 communities
Attack communities to evaluate: 14
Knowledge graph: 38 nodes, 43 edges


In [3]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(name="attack_techniques", embedding_function=ef)

if collection.count() == 0:
    print("Populating ChromaDB...")
    with open(STIX_FILE, "r", encoding="utf-8") as f:
        bundle = json.load(f)

    docs, ids, metas = [], [], []
    for obj in bundle.get("objects", []):
        if obj.get("type") != "attack-pattern" or obj.get("revoked"):
            continue
        tid = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                tid = ref.get("external_id")
                break
        if not tid:
            continue
        tactics = [
            p["phase_name"].replace("-", " ").title()
            for p in obj.get("kill_chain_phases", [])
            if p.get("kill_chain_name") == "mitre-attack"
        ]
        text = (
            f"ID: {tid}\n"
            f"Name: {obj.get('name', '')}\n"
            f"Tactic: {', '.join(tactics)}\n"
            f"Description: {obj.get('description', '')}"
        )
        docs.append(text)
        ids.append(tid)
        metas.append({"technique_id": tid, "name": obj.get("name", ""), "tactic": ", ".join(tactics)})

    collection.add(ids=ids, documents=docs, metadatas=metas)
    print(f"Inserted {collection.count()} ATT&CK technique descriptions")
else:
    print(f"ChromaDB ready: {collection.count()} technique descriptions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB ready: 703 technique descriptions


In [4]:
def build_rag_query(cid: int) -> tuple:
    group = community_df[community_df["community_id"] == cid]
    triples = community_triples.get(str(cid), [])

    triple_sentences = []
    for t in triples:
        subj = t.get("subject", "").replace("_", " ")
        rel  = t.get("relation", "").replace("_", " ")
        tgt  = t.get("target", "").replace("_", " ")
        if subj and rel and tgt:
            triple_sentences.append(f"{subj} {rel} {tgt}")
    triple_text = ". ".join(triple_sentences)

    community_entities = set()
    for t in triples:
        community_entities.add(t.get("subject", ""))
        community_entities.add(t.get("target", ""))

    related_edges = kg_edges[
        kg_edges["source"].isin(community_entities) |
        kg_edges["target"].isin(community_entities)
    ].sort_values("weight", ascending=False).head(3)

    graph_context = ""
    if not related_edges.empty:
        parts = [
            f"{row['source'].replace('_',' ')} {row['relation'].replace('_',' ')} {row['target'].replace('_',' ')}"
            for _, row in related_edges.iterrows()
        ]
        graph_context = "Recurring patterns: " + "; ".join(parts) + "."

    retrieval_query = f"{triple_text}. {graph_context}".strip()
    alert_context   = " ".join(group["alert_text"].dropna().head(3).tolist())

    return retrieval_query, alert_context

print(f"Example query for community {eval_cids[0]}:")
print(build_rag_query(eval_cids[0])[0])

Example query for community 0:
ftp client targets ftp port 21 service. ftp client scans http web server. http flood source floods http web server. http flood source targets http port 80 service. Recurring patterns: dns resolver scans http web server; ftp client floods network bandwidth; ftp client scans ftp port 21 service.


In [5]:
REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "technique_id": {"type": "string"},
        "tactic":       {"type": "string"},
        "summary":      {"type": "string"},
        "evidence":     {"type": "string"},
        "next_step":    {"type": "string"}
    },
    "required": ["technique_id", "tactic", "summary", "evidence", "next_step"]
}
report_grammar = LlamaGrammar.from_json_schema(json.dumps(REPORT_SCHEMA))

print(f"Loading {MODEL_PATH}...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,
    n_gpu_layers=-1,
    n_threads=8,
    n_batch=256,
    verbose=False,
    seed=RANDOM_SEED,
)
print("LLM ready.")

Loading ../models/qwen2.5-3b-instruct-q4_k_m.gguf...


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


LLM ready.


In [6]:
def generate_rag_report(query: str, alert_context: str, retrieved_docs: list, retrieved_metas: list) -> dict:
    allowed_ids = [m.get("technique_id", "") for m in retrieved_metas]
    allowed_str = ", ".join(allowed_ids)
    context_block = "\n\n".join([
        f"[{i+1}] {str(doc)[:MAX_CHARS_PER_DOC]}"
        for i, doc in enumerate(retrieved_docs)
    ])

    prompt = f"""[INST] You are a cybersecurity analyst. Write a concise incident report.

Choose technique_id from this list only: {allowed_str}

OBSERVED BEHAVIOR:
{query}

RAW ALERT CONTEXT:
{alert_context}

RETRIEVED ATT&CK TECHNIQUES:
{context_block}

Rules:
- summary: one sentence describing what the attacker did
- evidence: two or three specific observations from the alert context that support your choice
- next_step: one concrete analyst action

Return ONLY valid JSON. [/INST]"""

    out = llm(prompt, max_tokens=400, temperature=0,
               seed=RANDOM_SEED, grammar=report_grammar, repeat_penalty=1.1)
    raw = out["choices"][0]["text"].strip()

    try:
        result = json.loads(raw)
        if result.get("technique_id", "") not in allowed_ids:
            result["technique_id"] = "Unknown"
            result["evidence"] = "[Hallucinated ID corrected]"
        return result
    except json.JSONDecodeError:
        return {"technique_id": "Unknown", "tactic": "Unknown",
                "summary": "Parse error", "evidence": "N/A", "next_step": "Manual review"}


def generate_baseline_report(query: str) -> dict:
    prompt = f"""[INST] You are a cybersecurity analyst. Write a concise incident report.

OBSERVED BEHAVIOR:
{query}

Rules:
- summary: one sentence describing what the attacker did
- evidence: two or three specific observations that support your choice
- next_step: one concrete analyst action

Identify the most likely MITRE ATT&CK technique and return ONLY valid JSON. [/INST]"""

    out = llm(prompt, max_tokens=400, temperature=0,
               seed=RANDOM_SEED, grammar=report_grammar, repeat_penalty=1.1)
    raw = out["choices"][0]["text"].strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"technique_id": "Unknown", "tactic": "Unknown",
                "summary": "Parse error", "evidence": "N/A", "next_step": "Manual review"}

In [7]:
def parent_match(gt_id: str, gen_id: str) -> bool:
    if not gt_id or not gen_id or gen_id == "Unknown":
        return False
    return gt_id.split(".")[0] == gen_id.split(".")[0]


print("Warming up")
_ = llm("[INST] Test. [/INST]", max_tokens=5, temperature=0, seed=RANDOM_SEED)
print("Starting evaluation\n")

results = []

for cid in eval_cids:
    group = community_df[community_df["community_id"] == cid]
    ground_truth = group["attck_technique_id"].mode().iloc[0]

    query, alert_context = build_rag_query(cid)

    retrieval = collection.query(query_texts=[query], n_results=TOP_K)
    docs = retrieval["documents"][0]
    metas = retrieval["metadatas"][0]
    retrieved_ids = [m.get("technique_id", "") for m in metas]

    rag_report = generate_rag_report(query, alert_context, docs, metas)
    rag_id = rag_report.get("technique_id", "Unknown")

    base_report = generate_baseline_report(query)
    base_id = base_report.get("technique_id", "Unknown")

    retrieval_hit = ground_truth in retrieved_ids
    rag_grounded  = rag_id in retrieved_ids or rag_id == "Unknown"
    rag_exact     = (rag_id == ground_truth) and rag_grounded
    rag_parent    = parent_match(ground_truth, rag_id) and rag_grounded
    base_exact    = base_id == ground_truth
    base_parent   = parent_match(ground_truth, base_id)

    results.append({
        "community_id":     cid,
        "ground_truth":     ground_truth,
        "dominant_label":   group["Label"].mode().iloc[0],
        "dominant_tactic":  group["attck_tactic"].mode().iloc[0],
        "retrieved_ids":    retrieved_ids,
        "retrieval_hit":    retrieval_hit,
        "rag_technique_id": rag_id,
        "rag_grounded":     rag_grounded,
        "rag_exact_match":  rag_exact,
        "rag_parent_match": rag_parent,
        "rag_tactic":       rag_report.get("tactic", ""),
        "rag_summary":      rag_report.get("summary", ""),
        "rag_evidence":     rag_report.get("evidence", ""),
        "rag_next_step":    rag_report.get("next_step", ""),
        "base_technique_id":  base_id,
        "base_exact_match":   base_exact,
        "base_parent_match":  base_parent,
        "base_tactic":        base_report.get("tactic", ""),
        "base_summary":       base_report.get("summary", ""),
    })

    print(f"  Community {cid} [{ground_truth}] | "
          f"RAG: {rag_id} (exact={rag_exact}, parent={rag_parent}) | "
          f"Baseline: {base_id} (exact={base_exact}, parent={base_parent})")

print(f"\nEvaluation complete: {len(results)} communities")

Warming up
Starting evaluation

  Community 0 [T1046] | RAG: T1498.001 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 1 [T1110.001] | RAG: T1499.002 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 2 [BENIGN] | RAG: T1071.004 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 4 [T1110.001] | RAG: T1496.002 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 5 [T1110.001] | RAG: T1110.003 (exact=False, parent=True) | Baseline: T1078 (exact=False, parent=False)
  Community 6 [BENIGN] | RAG: T1571 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 10 [BENIGN] | RAG: T1071.004 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 11 [T1110.001] | RAG: T1499.002 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 12 [T1071.001] | RAG: T1498.001 (exa

In [8]:
n = len(results)

metrics = {
    "model":                      "qwen2.5-3b-instruct-q4_k_m",
    "n_communities_evaluated":    n,
    "top_k_retrieval":            TOP_K,
    "retrieval_hit_rate":         round(sum(r["retrieval_hit"]     for r in results) / n, 4),
    "rag_grounding_rate":         round(sum(r["rag_grounded"]      for r in results) / n, 4),
    "rag_exact_match_rate":       round(sum(r["rag_exact_match"]   for r in results) / n, 4),
    "rag_parent_match_rate":      round(sum(r["rag_parent_match"]  for r in results) / n, 4),
    "baseline_exact_match_rate":  round(sum(r["base_exact_match"]  for r in results) / n, 4),
    "baseline_parent_match_rate": round(sum(r["base_parent_match"] for r in results) / n, 4),
}

print("METRICS")
print(json.dumps(metrics, indent=2))

delta_exact  = metrics["rag_exact_match_rate"]  - metrics["baseline_exact_match_rate"]
delta_parent = metrics["rag_parent_match_rate"] - metrics["baseline_parent_match_rate"]

print(f"\nRAG exact:      {metrics['rag_exact_match_rate']:.2%}")
print(f"Baseline exact: {metrics['baseline_exact_match_rate']:.2%}")
print(f"Delta:          {delta_exact:+.2%}")
print(f"\nRAG parent:     {metrics['rag_parent_match_rate']:.2%}")
print(f"Baseline parent:{metrics['baseline_parent_match_rate']:.2%}")
print(f"Delta:          {delta_parent:+.2%}")

METRICS
{
  "model": "qwen2.5-3b-instruct-q4_k_m",
  "n_communities_evaluated": 14,
  "top_k_retrieval": 5,
  "retrieval_hit_rate": 0.1429,
  "rag_grounding_rate": 1.0,
  "rag_exact_match_rate": 0.0,
  "rag_parent_match_rate": 0.0714,
  "baseline_exact_match_rate": 0.0,
  "baseline_parent_match_rate": 0.0
}

RAG exact:      0.00%
Baseline exact: 0.00%
Delta:          +0.00%

RAG parent:     7.14%
Baseline parent:0.00%
Delta:          +7.14%


In [9]:
print("SAMPLE REPORTS: RAG vs BASELINE\n")
for r in results[:3]:
    print(f"Community {r['community_id']} | {r['dominant_label']} | {r['dominant_tactic']}")
    print(f"  Ground truth:  {r['ground_truth']}")
    print(f"  Retrieved:     {r['retrieved_ids']}")
    print(f"  [RAG]      {r['rag_technique_id']} | exact={r['rag_exact_match']} parent={r['rag_parent_match']}")
    print(f"             Summary:  {r['rag_summary']}")
    print(f"             Evidence: {r['rag_evidence']}")
    print(f"             Action:   {r['rag_next_step']}")
    print(f"  [Baseline] {r['base_technique_id']} | exact={r['base_exact_match']} parent={r['base_parent_match']}")
    print(f"             Summary:  {r['base_summary']}")
    print("=" * 60 + "\n")

SAMPLE REPORTS: RAG vs BASELINE

Community 0 | PortScan | Impact
  Ground truth:  T1046
  Retrieved:     ['T1499.002', 'T1071.002', 'T1498.001', 'T1567', 'T1071.001']
  [RAG]      T1498.001 | exact=False parent=False
             Summary:  An attacker is flooding the HTTP web server with traffic, causing a denial of service.
             Evidence: [Flow to HTTP] Duration 5382184us, Flags none; [Flow to HTTP-alt] Duration 68920us, Fwd pkts 4, Bwd pkts 3, Bytes/s 4933.26; [Flow to HTTP-alt] Duration 997272us, Fwd pkts 3, Bwd pkts 3, Bytes/s 18.05.
             Action:   Investigate the source of the flood traffic and identify any potential vulnerabilities in the web server.
  [Baseline] T1078 | exact=False parent=False
             Summary:  The attacker is scanning and flooding the HTTP web server.

Community 1 | FTP Patator | Credential Access
  Ground truth:  T1110.001
  Retrieved:     ['T1499.002', 'T1110.003', 'T1499.001', 'T1499', 'T1563.001']
  [RAG]      T1499.002 | exact=False p

In [10]:
pd.DataFrame(results).to_csv(REPORTS_FILE, index=False)

with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(f"Reports -> {REPORTS_FILE}")
print(f"Metrics -> {METRICS_FILE}")


Reports -> ../data/results/stage5_rag_reports.csv
Metrics -> ../data/results/stage5_rag_metrics.json
